# DeepWeeds Lab Day 2
Nguyen Hong Cuong - 2A202602415. Select every decision on validation; enable test export only after the final configuration is frozen.

In [ ]:
%pip -q install timm openpyxl scikit-learn matplotlib
import os, sys, platform, hashlib, subprocess, urllib.request, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

REPO_URL = 'https://github.com/hongcuong26-debug/K4-Track4-Day2-NguyenHongCuong-2A202602415-Deeplearning-Advance.git'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
candidates = [
    Path.cwd(),
    Path.cwd() / 'code',
    Path.cwd() / 'submissions/2A202602415_nguyen_hong_cuong/code',
    WORK_DIR / 'deepweeds-lab/submissions/2A202602415_nguyen_hong_cuong/code',
]
CODE_DIR = next((p.resolve() for p in candidates if (p / 'dataset.py').exists()), None)
if CODE_DIR is None:
    clone_dir = WORK_DIR / 'deepweeds-lab'
    if not clone_dir.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(clone_dir)], check=True)
    CODE_DIR = clone_dir / 'submissions/2A202602415_nguyen_hong_cuong/code'
REPO_DIR = CODE_DIR.parents[2]
assert (CODE_DIR / 'dataset.py').exists(), f'Missing dataset.py in {CODE_DIR}'
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(CODE_DIR))
print('python', platform.python_version(), 'torch', torch.__version__, 'timm', timm.__version__)
print('code:', CODE_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - enable a Kaggle GPU accelerator before training')

## Data integrity
The official archive MD5 is `b7b30f96d466fba86016aa5a26606e0f`. Keep the archive until this check passes.

In [ ]:
DATA_ROOT = WORK_DIR / 'deepweeds-data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
kaggle_input = Path('/kaggle/input')

# Reuse an attached Kaggle dataset when available; otherwise download once per session.
local_labels = REPO_DIR / 'data/labels/train_subset0.csv'
attached_labels = local_labels if local_labels.exists() else (next(iter(kaggle_input.rglob('train_subset0.csv')), None) if kaggle_input.exists() else None)
LABELS_DIR = attached_labels.parent if attached_labels else DATA_ROOT / 'labels'
LABELS_DIR.mkdir(parents=True, exist_ok=True)
label_base = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ('labels.csv', 'train_subset0.csv', 'val_subset0.csv', 'test_subset0.csv'):
    target = LABELS_DIR / name
    if not target.exists():
        urllib.request.urlretrieve(f'{label_base}/{name}', target)

local_archive = REPO_DIR / 'data/images.zip'
local_jpg = next((REPO_DIR / 'images').glob('*.jpg'), None)
attached_archive = local_archive if local_archive.exists() else (next(iter(kaggle_input.rglob('images.zip')), None) if kaggle_input.exists() else None)
attached_jpg = local_jpg if local_jpg is not None else (next(iter(kaggle_input.rglob('*.jpg')), None) if kaggle_input.exists() else None)
if attached_jpg is not None:
    IMAGES_DIR = attached_jpg.parent
    print('Using attached images:', IMAGES_DIR)
else:
    archive = attached_archive or (DATA_ROOT / 'images.zip')
    if not archive.exists():
        print('Downloading DeepWeeds images.zip (~492 MB)...')
        urllib.request.urlretrieve('https://zenodo.org/records/7939060/files/images.zip?download=1', archive)
    md5 = hashlib.md5()
    with archive.open('rb') as fh:
        for chunk in iter(lambda: fh.read(8 * 1024 * 1024), b''):
            md5.update(chunk)
    actual = md5.hexdigest()
    assert actual == 'b7b30f96d466fba86016aa5a26606e0f', actual
    print('MD5 OK')
    IMAGES_DIR = DATA_ROOT / 'images'
    if not next(IMAGES_DIR.glob('*.jpg'), None):
        IMAGES_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(IMAGES_DIR)
print('labels:', LABELS_DIR)
print('images:', IMAGES_DIR)

In [ ]:
from dataset import load_split, check_split
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_stats = check_split(train_df, val_df, test_df, IMAGES_DIR)
display(pd.DataFrame(split_stats['per_class']).rename_axis('Label'))
pd.DataFrame(split_stats['per_class']).plot.bar(figsize=(10, 4), title='DeepWeeds fold 0 class distribution')

## Backbone screen and controlled ablations

In [ ]:
from experiments import registry
backbone_configs = registry('backbones', backbone='convnext_tiny', seed=0)
[(c.exp_id, c.backbone) for c in backbone_configs]
# GPU run: from train import run; [run(c) for c in backbone_configs]

In [ ]:
# Replace with the backbone selected from validation results.
selected_backbone = 'convnext_tiny'
ablation_configs = registry('ablations', backbone=selected_backbone, seed=0)
[(c.exp_id, c.init, c.aug, c.loss, c.mix, c.ema_decay) for c in ablation_configs]
# GPU run: [run(c) for c in ablation_configs]

## Inference experiments
Use `inference.py` for flip/multicrop/multiscale TTA, probability or logit aggregation, ensembles, temperature scaling fitted only on validation, and Conv-BN fusion. Use `benchmark.py` for synchronized p50/p95/p99 latency.

In [ ]:
from inference import aggregate_views, fit_temperature, apply_temperature
from benchmark import latency_report
# Example after loading saved validation logits:
# T = fit_temperature(val_logits, val_labels)
# calibrated_test_probs = apply_temperature(test_logits, T)

## Final test protocol
Freeze the full configuration from validation. Run F01 and baseline T00 for seeds 0, 1, and 2 with `save_test_predictions=True`. Never revise the configuration after viewing test metrics.

In [ ]:
# Example only after validation selection:
# from train import Config, run
# for seed in (0, 1, 2):
#     run(Config(exp_id='F01', seed=seed, backbone=selected_backbone,
#                save_test_predictions=True, ...))
#     run(Config(exp_id='T00', seed=seed, backbone=selected_backbone,
#                save_test_predictions=True))

In [ ]:
# Run from the repository root after final predictions exist:
# !python eval.py score --pred 'submissions/2A202602415_nguyen_hong_cuong/predictions/F01_seed*_test.csv' --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out
# !python eval.py grade --final 'submissions/2A202602415_nguyen_hong_cuong/predictions/F01_seed*_test.csv' --baseline 'submissions/2A202602415_nguyen_hong_cuong/predictions/T00_seed*_test.csv' --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --out eval_out